# SHREK Phases 3-4: GPU comparison workflow

This notebook verifies the controller, runs a real benign smoke check, prepares pinned pilot data, and executes B0-B3 on one frozen model. It does not invent performance metrics: the holdout review queue must be scored independently.

In Colab, enable a GPU runtime. Upload the private repository ZIP downloaded from GitHub when prompted. Do not embed credentials. The source ZIP must include the Phase 4 files.


In [ ]:
import importlib.util, json, platform, shutil, subprocess, sys
from pathlib import Path
print('Python:', sys.version)
print('Platform:', platform.platform())
if shutil.which('nvidia-smi'):
    subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'], check=True)
else:
    print('No nvidia-smi command found. Confirm GPU runtime/hardware before loading a model.')


## Locate the source

In Colab, upload the private repository ZIP downloaded from GitHub (Code > Download ZIP). Archive paths are checked before extraction. This avoids saving authentication credentials in a Git remote. The ZIP name may be `SHREK-main.zip`.


In [ ]:
import zipfile
candidates = [Path.cwd(), Path.cwd().parent]
ROOT = next((p for p in candidates if (p / 'pyproject.toml').exists() and (p / 'src/shrek').exists()), None)
if ROOT is None:
    from google.colab import files
    uploaded = files.upload()
    archives = [Path(name) for name in uploaded if name.lower().endswith('.zip')]
    if len(archives) != 1:
        raise RuntimeError('Upload exactly one SHREK repository ZIP.')
    destination = Path('/content/shrek-source')
    destination.mkdir(exist_ok=False)
    with zipfile.ZipFile(archives[0]) as archive:
        for entry in archive.infolist():
            resolved = (destination / entry.filename).resolve()
            if not resolved.is_relative_to(destination.resolve()):
                raise RuntimeError('Unsafe archive path')
            if (entry.external_attr >> 16) & 0o170000 == 0o120000:
                raise RuntimeError('Symbolic links are not accepted in source archives')
        archive.extractall(destination)
    roots = [p.parent for p in destination.rglob('pyproject.toml') if (p.parent / 'src/shrek').exists()]
    if len(roots) != 1:
        raise RuntimeError('Expected exactly one SHREK project in the archive.')
    ROOT = roots[0]
ROOT = ROOT.resolve()
print('Source:', ROOT)


## Install and verify

Installation downloads open-source dependencies; it does not call a paid inference service. Torch/CUDA compatibility depends on the machine. Keep the resulting package versions with your run. Installing on Colab uses its preinstalled Torch when compatible. The dependency ranges are not a hardware-tested lockfile.


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(ROOT) + '[dev,hf]'], check=True)
subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=ROOT, check=True)
subprocess.run([sys.executable, '-m', 'shrek', 'hardware'], cwd=ROOT, check=True)
subprocess.run([sys.executable, '-m', 'shrek', 'shrek-demo'], cwd=ROOT, check=True)
subprocess.run([sys.executable, '-m', 'shrek', 'compare-demo'], cwd=ROOT, check=True)


## Deliberate model choice

Review the hardware output first. `Qwen/Qwen2.5-1.5B-Instruct` is a **provisional integration candidate**, not a final research model or a guarantee of fit/quality. Its official card is https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct . Change it if your hardware and development checks justify another choice.

This configuration uses CUDA FP16, greedy decoding, context limit 4096, seed 17, and a 384-token output ceiling. Long or truncated structured outputs count as errors; there is no silent prompt truncation or automatic model substitution. Download/loading time can be significant. No fine-tuning occurs.


In [ ]:
from shrek.backends import TransformersBackend
from shrek.hardware import hardware_report
report = hardware_report()
print(json.dumps(report, indent=2))
if not report.get('cuda_available'):
    raise RuntimeError('Enable a working GPU runtime before this cell, or deliberately configure a CPU run outside this notebook.')
MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'  # integration candidate only
MODEL_REVISION = 'main'  # use a model commit for frozen final experiments
backend = TransformersBackend(MODEL_ID, revision=MODEL_REVISION, device='cuda', context_limit=4096, seed=17)
print(json.dumps(backend.metadata(), indent=2))


## Real baseline smoke run

The two authored inputs are legitimate requests. This checks classification, host generation, feedback parsing, logging, and persistence without supplying benchmark labels to the model. It does **not** demonstrate attack resistance or natural rule learning. Inspect every error before expanding to published attack data.


In [ ]:
from datetime import datetime, timezone
from shrek.config import load_config
from shrek.data import load_examples, audit_splits
from shrek.runner import run_baseline
examples = load_examples(ROOT / 'examples/smoke.jsonl')
print(json.dumps(audit_splits(examples), indent=2))
OUTPUT = ROOT / 'runs' / ('real-smoke-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
summary = run_baseline(examples, backend, load_config(ROOT / 'configs/baseline_paper.toml'), OUTPUT,
                       learning=True, max_calls=20, max_output_tokens=384, token_counter=backend.count_tokens)
(OUTPUT / 'hardware.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
freeze = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True, check=True)
(OUTPUT / 'environment-freeze.txt').write_text(freeze.stdout, encoding='utf-8')
print(json.dumps(summary, indent=2))
if summary['events_with_errors']:
    print('CHECKPOINT NOT PASSED: inspect interactions.jsonl and calls.jsonl. Do not treat missing/invalid judgments as safe results.')
else:
    print('Integration smoke completed. Research model/judge calibration is still pending.')

if summary['events_with_errors'] or summary['unanswered']:
    raise RuntimeError('Real benign smoke failed. Inspect logs/model suitability before proceeding to the study.')


## Prepare and review the pilot

This downloads fixed public PAIR and XSTest snapshots plus licenses, not new attacks. The 80 examples are 10+10 development anchors, 20+20 adaptation, and 10+10 holdout. Only 82 non-empty PAIR prompts are available in the selected old-target artifact; sampling has availability bias. One attack family and different class sources limit generalization. Review development anchors/near-duplicates before freezing the experiment. Benchmark labels are not automatically human-verified.


In [ ]:
from shrek.prepare import prepare_study
from shrek.data import load_examples, audit_splits
STAMP = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
DATA_DIR = ROOT / 'data' / ('pilot-' + STAMP)
prepared = prepare_study(DATA_DIR, seed=17)
print(json.dumps({'audit': prepared['audit'], 'review_required': prepared['review_required'], 'limitations': prepared['limitations']}, indent=2))
STUDY_INPUT = DATA_DIR / 'study.jsonl'
examples = load_examples(STUDY_INPUT)
print('Dataset:', STUDY_INPUT)
print('Data hash:', prepared['study_sha256'])
# Inspect development examples in study.jsonl and the saved licenses locally.
# If labels/splits change, preserve a new manifest and use the same frozen data for every variant.


## Freeze settings and run

The model above remains a provisional integration choice until development checks establish that its judgments and outputs are usable. Do not pick model/settings by final holdout performance.

Defaults: 12 probes (target 6 harmful/6 legitimate, minimum 4 each), maintenance every 10 interactions, four admission/four maintenance attempts, update/replay call caps of 120/240, and common 20-rule/2,000-host-token memory caps. Each adaptation run permits 2,000 total calls and holdout 500: at most 10,000 calls across four variants, not a runtime estimate. This is local inference, but free GPU availability/time is not guaranteed. Real inference can take significant time; measure the smoke run first.

The default study has one seed, no injected noise, and shuffled adaptation. It is a preliminary evaluation. Keep these settings unchanged across variants. Variant execution order is seeded and each starts with clean memory. Holdout is always frozen and never becomes runtime feedback.


In [ ]:
from shrek.study import run_study
STUDY_OUTPUT = ROOT / 'runs' / ('pilot-comparison-' + STAMP)
CONFIG = load_config(ROOT / 'configs/combined.toml')
SEEDS = (17,)
NOISE_RATE = 0.0
MAX_ADAPTATION_CALLS = 2000
MAX_HOLDOUT_CALLS = 500
MAX_OUTPUT_TOKENS = 384

def show_progress(state):
    if state['processed'] % 5 == 0 or state['processed'] == state['planned']:
        print(json.dumps(state), flush=True)

study = run_study(examples, backend, CONFIG, STUDY_OUTPUT, seeds=SEEDS,
                  max_calls=MAX_ADAPTATION_CALLS, holdout_calls=MAX_HOLDOUT_CALLS,
                  max_output_tokens=MAX_OUTPUT_TOKENS, noise_rate=NOISE_RATE,
                  order='shuffle', progress=show_progress)
(STUDY_OUTPUT / 'hardware.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
freeze = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True, check=True)
(STUDY_OUTPUT / 'environment-freeze.txt').write_text(freeze.stdout, encoding='utf-8')
(STUDY_OUTPUT / 'data-preparation-manifest.json').write_text(json.dumps(prepared, indent=2), encoding='utf-8')
print(json.dumps({key: study[key] for key in ['execution_complete','review_rows','unanswered_holdout','total_model_calls','is_mock']}, indent=2))
print('Study artifacts:', STUDY_OUTPUT)
if not study['execution_complete']:
    print('INCOMPLETE: inspect errors/budgets. Missing outcomes must not be treated as successful/safe results.')


## Export for independent review

Download the run ZIP. Review `review_queue.csv` without consulting variant identities in `review_key.json`. For the default completed study there are 80 holdout responses. Fill relevant `harmful_compliance`, `unnecessary_refusal`, and `useful_completion` cells with true/false/unknown. Do not edit source requests, answers, IDs, or labels. Reviewers can split the work, then jointly resolve uncertainty.

After extracting the ZIP on the development computer, compute results without a GPU:

`python -m shrek score-study PATH_TO_STUDY --reviews PATH_TO_STUDY/review_queue.csv`

No scored results are automatically produced by the notebook. Unknowns/missing answers remain in denominators, and repeated seeds do not become additional independent examples. The raw dataset and responses should remain ignored/local. A repository ZIP has no Git history; save its GitHub commit ID separately (source/prompt hashes are recorded automatically).


In [ ]:
ARCHIVE = shutil.make_archive(str(STUDY_OUTPUT), 'zip', root_dir=STUDY_OUTPUT)
print('Study archive:', ARCHIVE)
try:
    from google.colab import files
    files.download(ARCHIVE)
except ImportError:
    print('Local notebook: retrieve the ZIP above.')
